In [2]:
# ================== COORDENADAS DESDE estaciones.csv ==================
import re
import pandas as pd
import numpy as np

ESTACIONES_CSV = "../datos/estaciones.csv"   # <- tu ruta

def _region_prefix(clave: str) -> str:
    """
    'SURESTE3' -> 'SURESTE', 'NORESTE2' -> 'NORESTE', 'CENTRO' -> 'CENTRO'
    Quita dígitos finales y underscores si los hubiera.
    """
    if clave is None:
        return ""
    s = str(clave).strip().upper()
    s = re.sub(r"[_\s]+", "", s)
    s = re.sub(r"\d+$", "", s)   # elimina dígitos al final
    return s

def load_coords_tables(path: str):
    """
    Lee estaciones.csv y devuelve:
      - coords_by_station: DataFrame(estacion=Clave_Estacion, lat, lon)
      - coords_by_region:  DataFrame(region, lat, lon)  (centroide por región)
    Columnas esperadas (flexible): Nombre_Estacion, Clave_Estacion, lat, lon
    """
    c = pd.read_csv(path)
    # normaliza nombres
    c.columns = [x.strip() for x in c.columns]
    # renombres mínimos si vinieran variantes
    ren = {}
    if "Clave_Estacion" not in c.columns:
        for col in c.columns:
            if col.lower() in ("clave", "clave_estacion", "id", "id_estacion"):
                ren[col] = "Clave_Estacion"
                break
    if "Nombre_Estacion" not in c.columns:
        for col in c.columns:
            if col.lower() in ("nombre", "nombre_estacion", "estacion"):
                ren[col] = "Nombre_Estacion"
                break
    if ren:
        c = c.rename(columns=ren)

    req = {"Clave_Estacion", "lat", "lon"}
    if not req.issubset(c.columns):
        raise ValueError(f"estaciones.csv debe contener columnas {req}, encontradas: {set(c.columns)}")

    # tabla a nivel estación (clave exacta)
    coords_by_station = (c[["Clave_Estacion", "lat", "lon"]]
                           .rename(columns={"Clave_Estacion":"estacion"})
                           .dropna(subset=["lat","lon"])
                           .copy())
    coords_by_station["estacion"] = coords_by_station["estacion"].astype(str).str.upper()

    # tabla a nivel región (centroide de todas las estaciones cuyo prefijo es esa región)
    tmp = c.copy()
    tmp["region"] = tmp["Clave_Estacion"].map(_region_prefix)
    coords_by_region = (tmp.groupby("region", as_index=False)
                          .agg(lat=("lat","mean"), lon=("lon","mean"))
                          .dropna(subset=["lat","lon"]))
    coords_by_region["region"] = coords_by_region["region"].astype(str).str.upper()

    return coords_by_station, coords_by_region

# Cárgalas UNA vez
COORDS_BY_STATION, COORDS_BY_REGION = load_coords_tables(ESTACIONES_CSV)

def join_geo(summary_df: pd.DataFrame, stations_col: str = "estacion") -> pd.DataFrame:
    """
    Intenta unir primero por estación exacta (Clave_Estacion).
    Si no hay match (o la columna luce como regiones), cae a unión por región-centróide.
    """
    df = summary_df.copy()

    # Heurística: si los valores son tipo 'SURESTE', 'NORESTE', parecen regiones;
    # si tienen dígitos al final ('SURESTE3'), parecen estaciones.
    sample = df[stations_col].astype(str).str.upper().head(50)
    looks_region = (sample.str.fullmatch(r"[A-ZÁÉÍÓÚÑ]+").mean() > 0.8)

    if not looks_region:
        # Intento 1: join por estación exacta
        left = df.copy()
        left[stations_col] = left[stations_col].astype(str).str.upper()
        geo = left.merge(COORDS_BY_STATION, how="inner", left_on=stations_col, right_on="estacion")
        if not geo.empty:
            return geo
        # si no hubo match, cae a región
        looks_region = True

    if looks_region:
        # normaliza a prefijo de región y une contra centroides
        reg_df = df.copy()
        reg_df["region"] = reg_df[stations_col].astype(str).str.upper().map(_region_prefix)
        geo = reg_df.merge(COORDS_BY_REGION, how="inner", on="region")
        if not geo.empty:
            # renombrar para que el resto del pipeline siga usando 'estacion'
            geo = geo.rename(columns={"region":"estacion"})
            return geo

    raise ValueError("No se pudieron asignar coordenadas: revisa claves/columnas en estaciones.csv y en tu resumen.")


In [3]:
# =========================================================
# % días excedidos (PM10, PM2.5, O3) desde dataset global wide horario
# =========================================================

# !pip -q install folium branca
import re, math
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
from branca.colormap import LinearColormap

# -------------------- Parámetros --------------------
GLOBAL_PATH = "../datos/datos_20_25.csv"  # tu archivo global
YEAR        = 2024
POLLUTANTS  = ["pm10", "pm25", "o3"]   # los 3 que quieres analizar
COORDS_CSV  = "../datos/estaciones.csv"   # si existe: columnas esperadas ~ ['estacion','lat','lon']

# Si NO tienes estaciones.csv, usa un dict como respaldo:
COORDS_FALLBACK = {
    "CE":  (25.686613, -100.316116),
    "NE":  (25.781950, -100.188390),
    "NE2": (25.741670, -100.302220),
    "NE3": (25.784997, -100.051886),
    "NO":  (25.796980, -100.317910),
    "NO2": (25.819672, -100.580521),
    "NO3": (25.819672, -100.580521),
    "NTE": (25.755500, -100.289600),
    "NTE2":(25.808333, -100.326667),
    "SE":  (25.671600, -100.214500),
    "SE2": (25.647222, -100.095833),
    "SE3": (25.591008, -100.001594),
    "SO":  (25.673250, -100.458130),
    "SO2": (25.657160, -100.402680),
    "SUR": (25.640000, -100.300000),
}

# Límites NOM (en las mismas unidades que tu CSV)
LIMITS = {
    "pm10": {"mode": "mean24h", "limit": 75.0, "title": "PM10 {year} – % de días excedidos"},
    "pm25": {"mode": "mean24h", "limit": 45.0, "title": "PM2.5 {year} – % de días excedidos"},
    "o3":   {"mode": "max1h",   "limit": 95.0, "title": "O₃ {year} – % de días con ≥1 h por encima"},
}

# -------------------- Helpers --------------------
def _norm(s: str) -> str:
    s = str(s).lower()
    s = (s.replace("á","a").replace("é","e").replace("í","i").replace("ó","o").replace("ú","u")
           .replace("pm2.5","pm25").replace("pm2_5","pm25"))
    return re.sub(r"[^a-z0-9]", "", s)

def completeness_threshold(nmax: int) -> int:
    return 1 if nmax <= 1 else math.ceil(0.75 * nmax)

def keep_valid_days(long_df: pd.DataFrame) -> pd.DataFrame:
    # 75% adaptativo por estación/día (misma regla que ya usabas)
    n_day = long_df.groupby(["estacion","fecha"], as_index=False).agg(n=("valor","size"))
    nmax  = n_day.groupby("estacion", as_index=False)["n"].max().rename(columns={"n":"n_max"})
    n_day = n_day.merge(nmax, on="estacion", how="left")
    n_day["min_validos"] = n_day["n_max"].apply(completeness_threshold)
    n_day["dia_valido"]  = n_day["n"] >= n_day["min_validos"]
    return long_df.merge(n_day[["estacion","fecha","dia_valido"]],
                         on=["estacion","fecha"], how="left").query("dia_valido").copy()

def station_percent_exceeded(valid_long: pd.DataFrame, mode: str, limit: float) -> pd.DataFrame:
    if mode == "mean24h":
        daily = (valid_long.groupby(["estacion","fecha"], as_index=False)
                 .agg(agregado=("valor","mean")))
    elif mode == "max1h":
        daily = (valid_long.groupby(["estacion","fecha"], as_index=False)
                 .agg(agregado=("valor","max")))
    else:
        raise ValueError("mode debe ser 'mean24h' o 'max1h'.")
    daily["excede"] = daily["agregado"] > limit
    st = (daily.groupby("estacion", as_index=False)
          .agg(dias_excedidos=("excede","sum"),
               dias_validos=("excede","size")))
    st["pct_exced"] = 100*st["dias_excedidos"]/st["dias_validos"].clip(lower=1)
    return st

def load_coords(coords_csv: str | None) -> pd.DataFrame:
    if coords_csv:
        try:
            c = pd.read_csv(coords_csv)
            c.columns = [col.strip().lower() for col in c.columns]
            # admite variantes comunes
            rename_map = {}
            if "region" in c.columns and "estacion" not in c.columns:
                rename_map["region"] = "estacion"
            if "latitude" in c.columns and "lat" not in c.columns:
                rename_map["latitude"] = "lat"
            if "longitude" in c.columns and "lon" not in c.columns:
                rename_map["longitude"] = "lon"
            c = c.rename(columns=rename_map)
            return c[["estacion","lat","lon"]].dropna()
        except Exception:
            pass
    # fallback a dict
    return pd.DataFrame({
        "estacion": list(COORDS_FALLBACK.keys()),
        "lat": [v[0] for v in COORDS_FALLBACK.values()],
        "lon": [v[1] for v in COORDS_FALLBACK.values()],
    })

# -------------------- Loader específico a tu CSV --------------------
def wide_hourly_to_long(GLOBAL_PATH: str, pollutant: str, year: int) -> pd.DataFrame:
    """
    Espera columnas: date (timestamp), region (estación), y columnas de contaminantes (PM10, PM2.5, O3, ...).
    Devuelve: timestamp, fecha, estacion, valor
    """
    df = pd.read_csv(GLOBAL_PATH)
    # normaliza nombres visibles en tu captura:
    df.columns = [c.strip() for c in df.columns]
    # mapea nombres clave
    date_col   = "date" if "date" in df.columns else [c for c in df.columns if c.lower() in ("fecha","datetime","fecha_hora")][0]
    region_col = "region" if "region" in df.columns else "estacion"

    # Derretimos SOLO el contaminante solicitado
    # nombres de columnas tal como aparecen en tu archivo (PM10, PM2.5, O3):
    pol_map_visible = {"pm10":"PM10", "pm25":"PM2.5", "o3":"O3"}
    pol_col = pol_map_visible[pollutant] if pol_map_visible[pollutant] in df.columns else pollutant.upper()

    melted = df[[date_col, region_col, pol_col]].copy()
    melted.rename(columns={date_col: "timestamp", region_col: "estacion", pol_col: "valor"}, inplace=True)

    # parseos
    melted["timestamp"] = pd.to_datetime(melted["timestamp"], errors="coerce")
    melted = melted.dropna(subset=["timestamp"])
    melted["fecha"] = melted["timestamp"].dt.date

    # año y numerificación
    melted = melted[melted["timestamp"].dt.year == year]
    melted["valor"] = pd.to_numeric(melted["valor"], errors="coerce")
    melted = melted.dropna(subset=["valor", "estacion"])

    # deja solo columnas estándar
    return melted[["timestamp","fecha","estacion","valor"]].copy()

# -------------------- Mapa por contaminante --------------------
def build_map_for(pol: str):
    pol_key = pol.lower()
    mode, limit = LIMITS[pol_key]["mode"], LIMITS[pol_key]["limit"]

    # 1) Datos
    long_df = wide_hourly_to_long(GLOBAL_PATH, pol_key, YEAR)
    valid   = keep_valid_days(long_df)
    st      = station_percent_exceeded(valid, mode, limit)
    geo     = join_geo(st, stations_col="estacion")  # usa estaciones.csv
    if geo.empty:
        raise SystemExit("No hay estaciones con coordenadas válidas.")

    # % columna
    def _ensure_pct_col(df: pd.DataFrame) -> str:
        for c in ["pct_exced", "pct_exceed", "pct", "pct_excedidos"]:
            if c in df.columns:
                return c
        if {"dias_excedidos", "dias_validos"}.issubset(df.columns):
            df["pct_exced"] = 100.0 * df["dias_excedidos"] / df["dias_validos"].clip(lower=1)
            return "pct_exced"
        raise KeyError("No encuentro columna de porcentaje ni puedo calcularla.")
    pct_col = _ensure_pct_col(geo)

    # 2) Heat weights
    geo["weight"] = (geo[pct_col].clip(lower=0.0)) / 100.0
    geo.loc[geo["weight"] == 0, "weight"] = 1e-3
    heat_data = geo[["lat","lon","weight"]].values.tolist()

    # 3) Mapa base (SOLO 2 basemaps)
    center = [25.68435, -100.31721]
    m = folium.Map(location=center, zoom_start=11, tiles=None, control_scale=True)

    # Basemaps (solo dos)
    folium.TileLayer(
        tiles="CartoDB positron",
        name="CartoDB Positron (Blanco)",
        show=True,
        attr="© OpenStreetMap contributors, © CARTO"
        ).add_to(m)
    folium.TileLayer(
        tiles="OpenStreetMap",
        name="OpenStreetMap",
        show=False,
        attr="© OpenStreetMap contributors"
        ).add_to(m)


    # 4) Heatmap overlay
    HeatMap(heat_data, radius=26, blur=16, min_opacity=0.20) \
        .add_to(folium.FeatureGroup(name=f"Heatmap {pol_key.upper()}", show=True).add_to(m))

    # 5) Colormap para círculos
    red_cmap = LinearColormap(
        colors=['#fff5f0','#fee0d2','#fcbba1','#fc9272','#fb6a4a','#ef3b2c','#cb181d','#99000d'],
        vmin=0, vmax=float(geo[pct_col].max())
    )
    red_cmap.caption = f"{pol_key.upper()} {YEAR} – % de días excedidos"
    red_cmap.add_to(m)

    # 6) Estaciones: círculo con hover + etiqueta fija
    # === Estaciones (robusto): CircleMarker + tooltip hover + popup + etiqueta fija ===
    from folium import Tooltip, Popup
    from folium.features import DivIcon

    pct_col = "pct_exced" if "pct_exced" in geo.columns else "pct_exceed"
    station_layer = folium.FeatureGroup(name="Estaciones (círculos por intensidad)", show=True).add_to(m)

    for _, r in geo.iterrows():
        color = red_cmap(float(r[pct_col]))

        # Círculo interactivo (hover: muestra %; click: popup)
        cm = folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=8,
            weight=1,
            color="black",
            fill=True,
            fill_color=color,
            fill_opacity=0.95,
        ).add_to(station_layer)

        # Tooltip SOLO al pasar el mouse (porcentaje)
        Tooltip(
            f"{r['estacion']}: {r[pct_col]:.1f}% de días excedidos",
            sticky=True, direction="top", opacity=0.95,
        ).add_to(cm)

        # Popup al click con detalle
        Popup(
            f"<b>{r['estacion']}</b><br>"
            f"{pol_key.upper()} {year if 'year' in locals() else YEAR}<br>" # type: ignore
            f"Días excedidos: {int(r['dias_excedidos'])} / {int(r['dias_validos'])}<br>"
            f"% excedidos: {r[pct_col]:.1f}%",
            max_width=260
        ).add_to(cm)

        # Etiqueta SIEMPRE visible (nombre), sin bloquear eventos
        folium.Marker(
            [r["lat"], r["lon"]],
            icon=DivIcon(
                html=f"""
                <div style="
                    font-size:10px; font-weight:600; color:#111;
                    text-shadow:0 0 3px #fff;
                    transform: translate(-50%, -16px);
                    pointer-events: none;">
                    {r['estacion']}
                </div>
                """
            )
        ).add_to(station_layer)


    # 7) Anillos max/min
    row_max = geo.loc[geo[pct_col].idxmax()]
    row_min = geo.loc[geo[pct_col].idxmin()]
    folium.Circle([row_max["lat"], row_max["lon"]], radius=900, color="#cb181d", weight=3, fill=False,
                  tooltip=f"MAX: {row_max['estacion']} ({row_max[pct_col]:.1f}%)").add_to(m)
    folium.Circle([row_min["lat"], row_min["lon"]], radius=900, color="#2c7fb8", weight=3, fill=False,
                  tooltip=f"MIN: {row_min['estacion']} ({row_min[pct_col]:.1f}%)").add_to(m)

    # 8) Título
    title_html = f'''
    <h3 style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
               z-index: 9999; background: rgba(255,255,255,0.95); padding: 6px 10px;
               border-radius: 8px; font-family: sans-serif; font-weight: 700;">
      {pol_key.upper()} {YEAR} – % de días excedidos
    </h3>'''
    m.get_root().html.add_child(folium.Element(title_html))

    # 9) LayerControl SOLO UNA VEZ (al final)
    for k in list(m._children):
        if "layer_control" in k:
            del m._children[k]
    folium.LayerControl(collapsed=False).add_to(m)

    print(f"{pol_key.upper()} {YEAR} – mapa listo. Estaciones: {len(geo)}")
    return m


# -------------------- Ejecuta para los 3 --------------------
maps = {}
for pol in POLLUTANTS:
    maps[pol] = build_map_for(pol)

# Para ver un mapa en notebooks interactivos, muestra, p. ej.:
# maps["pm10"]


PM10 2024 – mapa listo. Estaciones: 15
PM25 2024 – mapa listo. Estaciones: 14
O3 2024 – mapa listo. Estaciones: 15


In [5]:
maps["pm10"]

In [ ]:
import os

outdir = "../mapas/mapas_html"
os.makedirs(outdir, exist_ok=True)

YEARS = [2020, 2024]

for year in YEARS:
    for pol in POLLUTANTS:  # ["pm10", "pm25", "o3"]
        # reconstruye el mapa para ese año
        fmap = build_map_for(pol) if YEAR == year else build_map_for(pol)
        
        # guarda con nombre dinámico
        fname = f"mapa_{pol.upper()}_{year}.html"
        path = os.path.join(outdir, fname)
        fmap.save(path)
        print(f"Mapa guardado en: {path}")


PM10 2020 – mapa listo. Estaciones: 14
Mapa guardado en: mapas_html/mapa_PM10_2020.html
PM25 2020 – mapa listo. Estaciones: 14
Mapa guardado en: mapas_html/mapa_PM25_2020.html
O3 2020 – mapa listo. Estaciones: 13
Mapa guardado en: mapas_html/mapa_O3_2020.html
PM10 2020 – mapa listo. Estaciones: 14
Mapa guardado en: mapas_html/mapa_PM10_2024.html
PM25 2020 – mapa listo. Estaciones: 14
Mapa guardado en: mapas_html/mapa_PM25_2024.html
O3 2020 – mapa listo. Estaciones: 13
Mapa guardado en: mapas_html/mapa_O3_2024.html
